# 项目问题记录

## Chroma 通常会把数据存在内存里，程序一关数据就没了

## 为什么数据库用 PostgreSQL而不是mysql? 核心原因：要做向量相似度检索，MySQL 原生不方便，PostgreSQL + pgvector 正好合适。

## ANN 近似检索全称是 Approximate Nearest Neighbor Search（近似最近邻搜索），是向量检索里非常核心的一种技术。
在海量高维向量里找“和最相似的那几个向量”时，不保证 100% 精确到绝对最近邻，而是用可接受的一点精度损失，换取极大的检索速度和内存效率提升。
ANN 的基本思路
通过一些预处理结构，把向量“分组 / 索引化”，检索时只在一个或少数几个候选集合里算距离，而不是遍历全部向量：
常见方法包括：
基于量化 / 聚类
比如 Faiss 的 IVF（倒排文件索引）：先把向量聚成若干类中心，检索时先找最近的几个类中心，只在这几类里细算。
基于图
比如 HNSW（Hierarchical Navigable Small World）：构建多层图结构，在图里“跳着走”，快速逼近最近邻。
基于哈希
局部敏感哈希（LSH）：让相似的向量有更高概率落到同一个哈希桶里，检索时只看同桶向量。
你前面用到的 Chroma、Faiss、Milvus、Weaviate​ 等向量数据库，底层默认基本都是 ANN 检索。
ANN 在 LangChain / RAG 里的体现
当你写：
search_kwargs={"k": 5}
背后向量库做的通常不是暴力 KNN，而是 ANN：
从成千上万条摘要/文档向量里
近似找出 top‑5 最相似的
再返回给 MultiVectorRetriever
这对大规模文档集非常关键，否则一查就卡住。
ANN 的关键指标
召回率（Recall）：找回的结果里包含真实最近邻的比例，ANN 通常可做到 90%+ 甚至 95%+
QPS（每秒查询数）：速度提升往往几十倍到上百倍
索引构建开销：ANN 通常需要先建索引（训练聚类、构图等）

## ANN 的核心就在于如何高效地构建和定位候选集，从而避免全局遍历。
不同 ANN 算法确定候选集的方式完全不同，主要有以下几种经典策略：
1. 基于聚类/划分的方法（IVF - 倒排文件索引）
原理：先把所有向量用 K-means 等聚类算法分成 N 个簇（Voronoi 单元），每个簇有一个质心向量。
候选集确定过程：
建索引时：对所有向量做聚类，每个向量归属到离它最近的质心所在的簇，记录每个簇的向量列表。
检索时：
计算查询向量与所有质心的距离
选出距离最近的 nprobe 个簇（比如 nprobe=10）
候选集 = 这 10 个簇内的所有向量
最终：只在这些候选向量里做精确距离计算，排序取 top-k
候选集大小 ≈ nprobe × (总向量数 / 总簇数)
调整方式：增大 nprobe → 候选集变大 → 召回率提高但速度下降。
2. 基于图的方法（HNSW - Hierarchical Navigable Small World）
原理：构建多层图结构，上层稀疏（长跳），下层稠密（精细）。
候选集确定过程（没有显式的“候选集集合”）：
从最高层的入口点开始
在当前层贪心地走到离查询向量最近的节点
进入下一层，重复这个过程
到达最底层后，在该层周围探索 efSearch 个邻居节点
候选集 = 这 efSearch 个邻居节点
这里的“候选集”是动态探索出来的，不是预先划分好的固定集合。
调整方式：增大 efSearch → 探索范围更大 → 召回率提高但速度下降。
3. 基于哈希的方法（LSH - Locality Sensitive Hashing）
原理：设计一组哈希函数，使得相似的向量有更高概率落在同一个哈希桶里。
候选集确定过程：
建索引时：用 L 个哈希函数族，每个向量经过哈希后落入某些桶，记录桶号到向量列表的映射。
检索时：
对查询向量计算同样的哈希值
找到它所在的哈希桶
候选集 = 同桶内的所有向量
（有时还会加上相邻桶的向量）
调整方式：增加哈希表数量或加宽桶范围 → 候选集变大 → 召回率提高但速度下降。


## BM25 关键词搜索是什么
BM25（Best Matching 25）是一种基于统计的关键词匹配检索算法，属于传统信息检索方法，现在是 Elasticsearch、Solr 等搜索引擎的默认评分模型之一。
核心思想
不关心语义，只关心词是否出现、出现频率、在文档中的重要程度。
对查询里的关键词，在每篇文档里算一个相关性分数，按分数排序返回结果。
主要考虑的因素
词频（TF）：某个关键词在文档里出现次数越多，通常越相关，但会做饱和处理（出现太多也不再无限加分）。
逆文档频率（IDF）：关键词如果在很多文档里都出现（比如“的”“我们”），区分度低，权重就小；只在少数文档里出现的词权重高。
文档长度归一化：避免长文档单纯因为词多而得分高，会对长度做惩罚。
优点
可解释性强：为什么这条排前面，看关键词匹配情况就知道。
对专有名词、精确术语、代码、编号、法律条文等检索效果好。
速度快，不需要嵌入模型，算力要求低。
缺点
不理解语义：
查“人工智能”，它不会返回含“AI”“机器学习”的文档（除非显式扩展同义词）。
查“如何养猫”，匹配不到“猫咪饲养指南”。
对拼写错误、表述差异敏感。
难以处理多义、近义、抽象概念。

## 本项目用 pgvector 的向量距离检索；若要上 HNSW，是在 PostgreSQL 建 USING hnsw 索引，应用仍调用 ORDER BY embedding <-> query LIMIT k，并不在 Python 里手写 HNSW。

## 用户提问后界面只有 terminate {} 工具调用，没有助手正文；后端日志显示模型第一次就调了空 terminate，然后任务结束。
定位： 沿 Agent 主循环看——THINK → 有 tool_calls 就 EXECUTE → 命中 terminate 就 FINISHED。对照日志发现：模型没有生成 content，只发了空参数的 terminate，循环被提前终止。
**terminate是Agent里的一个结束任务工具，不是系统进程、也不是 HTTP 接口。**

它是什么
给大模型看的一个 function calling 工具，名字叫 terminate。模型觉得「这轮任务做完了」时可以调用它，告诉 Agent 循环：可以停了。

在代码里大致是：

schema：告诉模型「有个叫 terminate 的工具，用来结束」
handler：真正被调用时执行的逻辑（现在会校验并返回 message）
循环里的处理：执行到有效的 terminate 后，把状态设为 FINISHED**
根因（两层）：

模型侧： tool_choice=auto 且暴露了结束工具时，模型容易「有工具就调工具」，简单问题也会误调 terminate。
设计侧： terminate 描述过宽、参数为空；执行层「见到 terminate 就结束」，不校验是否已产出对用户可见的回答。结束信号被当成完成条件，回答步骤被跳过。
修复思路：

terminate 改为必填 message（最终回答）
空 terminate 不结束；必要时禁用工具再补一轮纯文本生成
提示词明确：能直接答就用文本，禁止空调用 terminate
收获： Agent 的「结束条件」不能只信任模型的工具调用，要和「是否已有可展示答案」绑定；工具 schema / 描述会强烈影响模型行为，结束类工具尤其要谨慎

面试官常追问 & 短答
Q：为什么不用「无 tool_calls 就结束」，还要 terminate？
A：很多 ReAct/Manus 风格 Agent 用显式结束工具收尾。但若循环已支持「无工具=结束」，terminate 就是冗余且高风险的；要么去掉，要么强制携带最终答案。

Q：这算模型问题还是工程问题？
A：两边都有。模型有工具偏好；工程若不对结束条件做校验，就会把误调用放大成「无回复」线上故障。

Q：怎么证明修对了？
A：复现原日志路径——简单问答不应再空结束；若仍误调空 terminate，应走强制文本补全并落库/SSE 推送可见回答。

Q：同类还要注意什么？
A：工具描述要窄、必填参数要清；结束/删除等高风险工具要二次校验；关键路径打日志（tool name + arguments + content 是否为空）。

# Agent 设计模式 
**Agent 设计模式（Agent Design Patterns）**，是指在构建“智能体（Agent）系统”时反复出现、被验证有效的一类架构与交互范式。它们本质上类似于软件工程中的设计模式（如MVC、观察者模式），但专门面向**具备感知、推理、决策与行动能力的系统**，尤其是在以大模型（LLM）为核心的系统中非常常见。
---

## 一、先把“Agent”说清楚

在AI系统中，一个 Agent 通常具备四个基本组件：

* **Perception（感知）**：接收输入（文本、图像、API结果等）
* **Reasoning（推理）**：基于上下文进行分析（通常由LLM完成）
* **Planning（规划）**：拆解任务、决定步骤
* **Action（行动）**：调用工具/API、执行操作

👉 可以抽象为一个闭环：

```
Observation → Thought → Action → Observation ...
```

---

## 二、为什么需要“设计模式”

如果你只是简单调用一次 LLM，那不需要模式；
但一旦涉及：

* 多步骤推理（multi-step reasoning）
* 工具调用（tool use / function calling）
* 长期任务（long-horizon tasks）
* 多Agent协作（multi-agent systems）

就会出现复杂性爆炸，这时候就需要结构化模式来约束系统行为。

---

## 三、常见 Agent 设计模式（核心）

### 1. ReAct 模式（Reason + Act）

最经典的Agent模式之一，由论文提出。

**核心思想：推理和行动交替进行**

流程：

```
Thought → Action → Observation → Thought → ...
```

特点：

* 每一步都显式“思考”
* 决定是否调用工具（如搜索、数据库）

适用场景：

* 需要外部信息（检索、计算）
* 多步任务

---

### 2. Plan-and-Execute（规划-执行分离）

把“思考”和“执行”彻底拆开：

* **Planner**：一次性生成完整计划
* **Executor**：按步骤执行

优点：

* 结构清晰
* 可控性强（适合工程化）

缺点：

* 计划一旦错，后续全部偏离

---

### 3. Reflection / Self-Refine（反思模式）

Agent 在执行后进行自我评估和修正。

结构：

```
生成结果 → 自我评估 → 修改 → 输出
```

典型用途：

* 提高生成质量（论文写作、代码生成）
* 减少 hallucination

---

### 4. Tool-Using Agent（工具调用模式）

Agent 不直接回答，而是：

* 调用工具（搜索、计算器、数据库）
* 再整合结果

本质是：

> LLM = 控制器（controller），工具 = 执行器（executor）

---

### 5. Multi-Agent（多智能体协作）

多个Agent分工合作，例如：

* Planner Agent（规划）
* Research Agent（检索）
* Critic Agent（评估）

典型结构：

* 辩论（debate）
* 角色分工（role-based collaboration）

---

### 6. Memory-Augmented Agent（带记忆的Agent）

引入长期记忆：

* Short-term memory（上下文）
* Long-term memory（向量数据库等）

解决：

* 上下文长度限制
* 跨任务持续性

---

### 7. Hierarchical Agent（分层Agent）

类似操作系统调度：

* 高层：战略决策
* 低层：具体执行

适用于复杂任务（如自动科研、自动编程）

---

## 四、从工程视角总结（你可以这样理解）

如果用系统设计语言描述：

| 模块         | 对应组件              |
| ---------- | ----------------- |
| LLM        | 决策核心（Policy）      |
| Prompt     | 控制逻辑（Program）     |
| Tools      | 外部执行接口            |
| Memory     | 状态存储              |
| Agent Loop | 控制流（Control Flow） |

👉 本质上：

> Agent 设计模式 = “如何组织 LLM + 工具 + 状态 + 控制流”

---

## 一句话总结

**Agent 设计模式 = 构建“会思考 + 会行动”的AI系统时，用来组织推理流程、工具调用和任务执行的标准化结构方法。**



# Plan-and-Execute 和 ReAct是什么
这两个是当前 **LLM Agent 系统中最核心的两种控制范式**，差别本质在于：
👉 **“先整体规划再执行” vs “边想边做、动态决策”**

我直接从**机制、流程、优缺点、适用场景**给你拆清楚。

---

# 一、ReAct（Reason + Act）

## 1）核心思想

ReAct = **推理（Reasoning）+ 行动（Acting）交替进行**

Agent 每一步都做三件事：

```text
Thought → Action → Observation → Thought → ...
```

* **Thought**：当前要怎么做（LLM生成）
* **Action**：调用工具（搜索/API/数据库等）
* **Observation**：工具返回结果

👉 是一个**闭环、在线决策（online decision-making）过程**

---

## 2）直观例子

任务：查某疾病的最新治疗方案

ReAct 会这样走：

```text
Thought: 我需要查最新论文
Action: 搜索 PubMed
Observation: 返回论文列表

Thought: 选择最相关的一篇
Action: 打开论文摘要
Observation: 得到摘要

Thought: 总结治疗方案
Action: 输出答案
```

---

## 3）特点

**优点：**

* 灵活（随时根据新信息调整）
* 对不确定任务很强
* 类似人类“边查边想”

**缺点：**

* 推理路径可能不稳定
* 容易走弯路（探索成本高）
* 难以严格控制流程

---

## 4）本质（更技术一点）

ReAct ≈ **带外部工具的自回归决策策略**

可以理解为：

> 一个在环境中不断更新 belief state 的 policy

---

# 二、Plan-and-Execute（规划-执行）

## 1）核心思想

把任务分成两个阶段：

### 阶段1：Planner（规划器）

一次性生成完整计划

### 阶段2：Executor（执行器）

按步骤执行，不再重新规划

---

## 2）流程

```text
User Query
   ↓
Planner: 生成 Plan（步骤1,2,3,...）
   ↓
Executor: 按顺序执行每一步
   ↓
输出结果
```

👉 是一个**离线规划（offline planning）过程**

---

## 3）直观例子（同一个任务）

```text
Plan:
1. 搜索PubMed
2. 筛选最新论文
3. 阅读摘要
4. 总结治疗方案

Execute:
→ Step1 执行搜索
→ Step2 筛选
→ Step3 阅读
→ Step4 总结
```

---

## 4）特点

**优点：**

* 结构清晰（适合工程实现）
* 可控性强（流程固定）
* 更容易调试（每一步可观测）

**缺点：**

* 计划如果错了 → 全部执行错误
* 对动态环境适应差
* 不够灵活

---

## 5）本质（更技术一点）

Plan-and-Execute ≈

> **先生成一个完整 trajectory（轨迹），再执行**

类似：

* classical planning
* task decomposition

---

# 三、核心对比（重点）

| 维度   | ReAct            | Plan-and-Execute |
| ---- | ---------------- | ---------------- |
| 决策方式 | 在线（step-by-step） | 离线（先规划）          |
| 灵活性  | 高                | 低                |
| 稳定性  | 较低               | 高                |
| 可控性  | 较弱               | 强                |
| 适合任务 | 不确定、开放问题         | 结构化、多步骤任务        |
| 错误传播 | 局部               | 全局（plan错=全错）     |

---

# 四、一句话对比（非常关键）

* **ReAct：**
  👉 “我边做边想”

* **Plan-and-Execute：**
  👉 “我先想好再做”

---

# 五、在你研究方向里的启发（重要）

你在做 **multimodal + concept bottleneck**，这里其实可以直接用：

### 用 ReAct：

* 动态决定：

  * 是否使用图像特征
  * 是否依赖文本
  * 哪些 concept 更重要

👉 类似：

```text
Thought: 图像信息不够 → 查询 clinical text
```

---

### 用 Plan-and-Execute：

* 固定 pipeline：

```text
Plan:
1. 提取视觉concept
2. 编码文本
3. 融合
4. 分类
```

👉 更接近你现在的模型结构


# 七、总结

**ReAct** 和 **Plan-and-Execute** 的本质差别：

> ReAct 是“动态策略”，
> Plan-and-Execute 是“静态策略”。


# Plan-and-Execute 和 ReAct 在工具调用失败时的回退策略差异
这是一个**很关键但常被忽略的工程差异点**：
👉 两种范式在**工具调用失败（tool failure）**时，本质上体现的是
**“局部自适应恢复” vs “全局重规划/刚性执行”** 的区别。

下面从机制层面拆清楚。

---

# 一、先定义“工具调用失败”

在 Agent 系统里，failure 不只是报错，还包括：

* API error（超时 / 500 / 权限）
* 返回结果为空或低质量
* 结果与预期语义不匹配（semantic failure）
* 工具不可用（tool unavailable）

---

# 二、ReAct 的回退策略（局部闭环恢复）

## 1）核心机制

ReAct 是一个 **闭环控制系统**：

```text
Thought → Action → Observation → Thought ...
```

👉 **失败会直接进入下一轮 Thought，被模型“吸收”**

---

## 2）回退行为模式

当工具失败时：

```text
Observation: ERROR / empty / irrelevant
↓
Thought:
  - 分析失败原因
  - 选择替代策略
↓
Action:
  - 重试 / 换工具 / 改查询
```

---

## 3）典型策略

### （1）Retry（重试）

```text
Thought: 可能是查询太模糊 → 改写 query
```

### （2）Tool Switching（工具切换）

```text
Thought: 搜索API失败 → 改用另一个搜索源
```

### （3）Decomposition（任务重拆）

```text
Thought: 任务太复杂 → 先查子问题
```

### （4）Fallback to reasoning（退化为纯推理）

```text
Thought: 工具不可用 → 依赖已有知识回答
```

---

## 4）本质

ReAct 的回退策略可以形式化为：

> **Policy π(a | s)** 在失败后更新 belief state 再决策

👉 是一种 **online error recovery（在线纠错）**

---

## 5）优缺点

**优点：**

* 强鲁棒性（robustness）
* 局部修复，不影响整体流程
* 可处理未知错误

**缺点：**

* 可能陷入循环（retry loop）
* 成本不可控（多次调用）
* 行为不稳定

---

# 三、Plan-and-Execute 的回退策略（结构化/全局）

## 1）核心机制

```text
Plan → Execute step1 → step2 → ...
```

👉 执行阶段**默认不重新规划**

---

## 2）工具失败时的三种典型策略

---

### （1）Rigid Execution（刚性执行）

最简单（也是最差）：

```text
Step2 失败 → 直接报错 / 跳过
```

👉 无真正“回退能力”

---

### （2）Local Retry（局部重试）

Executor 层处理：

```text
if tool_fail:
    retry(k times)
```

特点：

* 不改变 plan
* 只做参数级修复

👉 类似传统系统的 retry policy

---

### （3）Replanning（重新规划）【关键】

当失败严重时：

```text
Execution failure
   ↓
触发 Planner
   ↓
生成新 Plan
   ↓
重新执行
```

👉 这是 Plan-and-Execute 的“真正回退”

---

## 3）问题点

### （1）错误传播（error propagation）

* 如果 plan 本身错了：
  👉 所有步骤都错

---

### （2）回退成本高

* 需要重新规划 entire trajectory
* 比 ReAct 重很多

---

### （3）上下文割裂

* Planner 和 Executor 分离
* failure 信息可能丢失细节

---

## 4）本质

Plan-and-Execute 的回退是：

> **plan-level correction（轨迹级修复）**

而不是 step-level adaptation

---

# 四、核心差异总结（重点）

| 维度     | ReAct              | Plan-and-Execute      |
| ------ | ------------------ | --------------------- |
| 回退粒度   | **step-level（局部）** | **plan-level（全局）**    |
| 失败处理   | 融入下一轮推理            | 触发 retry 或 replanning |
| 灵活性    | 高                  | 中/低                   |
| 稳定性    | 低                  | 高                     |
| 成本控制   | 难                  | 易                     |
| 错误影响范围 | 局部                 | 全局                    |

---

# 五、一个直观类比（非常有帮助）

### ReAct：

像一个人在做实验：

> 失败 → 立刻调整 → 继续试

---

### Plan-and-Execute：

像工程施工：

> 先画图纸 → 按图施工
> 出问题 → 要么修补，要么重画图纸

---

# 六、工程实践中的改进（重要）

现实系统通常不会用“纯版本”，而是：

## Hybrid 回退策略

### 1）Plan + ReAct Execution

* 高层：Plan
* 低层：每一步用 ReAct

👉 失败时：

* 小问题 → ReAct 修
* 大问题 → Replan

---

### 2）带 Critic 的回退

引入评估器：

```text
Result → Critic → 判断失败 → 触发回退
```

---

### 3）Failure-aware planning

在 plan 中加入：

```text
if step fails:
    fallback strategy
```
---

# 八、一句话总结

> **ReAct 用“即时推理”修复错误，
> Plan-and-Execute 用“结构重构”修复错误。**


# LangChain 里 RunnableConfig 传递丢失、multi‑modal tool 导致 callback 堆栈溢出、agent scratchpad 越滚越长把上下文撑爆

这三个问题本质上是同一个根因：**Agent 执行链路没有把“控制状态”和“模型上下文”隔离好**。

## 1. RunnableConfig 传递丢失

**典型根因：**你在自定义 `RunnableLambda`、tool、wrapper、async 函数里没有继续把 `config` 传给下游 runnable。

LangChain 的 `RunnableConfig` 用来传递 `callbacks`、`tags`、`metadata`、`run_name`、`max_concurrency` 等运行时信息；官方 API 说明里也强调 callbacks 会作用于当前调用及子调用。([LangChain 参考文档][1])

错误写法：

```python
def my_step(x):
    return llm.invoke(x)   # config 丢了
```

推荐写法：

```python
from langchain_core.runnables import RunnableLambda

def my_step(x, config=None):
    return llm.invoke(x, config=config)

chain = RunnableLambda(my_step)
```

如果是 async：

```python
async def my_step(x, config=None):
    return await llm.ainvoke(x, config=config)
```

**原则：**凡是你手写了 wrapper，就显式接收并透传 `config`。

---

## 2. multi-modal tool 导致 callback 堆栈溢出

**典型根因：**tool 内部又调用 agent / chain / LLM，并且复用了父级 callbacks，造成递归 tracing 或 callback nesting 失控。

多模态 tool 更容易触发，因为它常见流程是：

```text
Agent → Tool(image+text) → Vision LLM → Tool callback → Agent callback → ...
```

修复策略：

```python
def multimodal_tool(input, config=None):
    child_config = {
        **(config or {}),
        "callbacks": None,   # 或换成精简 handler
        "tags": ["multimodal_tool"],
    }
    return vision_chain.invoke(input, config=child_config)
```

更稳妥的是不要让 tool 里再跑完整 agent，只跑**纯函数式子链**：

```text
Agent
 └── Tool
      └── vision_model / parser / retriever
```

避免：

```text
Agent
 └── Tool
      └── Agent
           └── Tool
```

尤其不要在 callback handler 的 `on_tool_start/on_chain_end` 里再次触发 runnable，否则很容易形成 callback recursion。

---

## 3. agent scratchpad 越滚越长撑爆上下文

**典型根因：**ReAct agent 会把 `intermediate_steps` / `agent_scratchpad` 持续塞回 prompt。工具返回大文本、图片描述、检索结果时，scratchpad 会指数级膨胀。

LangChain classic 的 `AgentExecutor` 有 `trim_intermediate_steps`，默认 `-1` 表示不裁剪。([LangChain 参考文档][2])

最直接处理：

```python
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    max_iterations=6,
    trim_intermediate_steps=3,
)
```

更推荐自定义裁剪，只保留最近几步 + 摘要：

```python
def trim_steps(steps):
    return steps[-3:]

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    trim_intermediate_steps=trim_steps,
)
```

对于 tool output，不要原样塞入 scratchpad。改成：

```python
def compress_tool_output(result: str, max_chars=1500):
    return result[:max_chars]
```

更工程化的做法：

```text
完整 tool result → 存外部 state / vector store / object store
scratchpad → 只存 result_id + 摘要 + 关键信号
```

---

## 推荐架构

```text
User Input
   ↓
Planner / Agent
   ↓
Tool Router
   ↓
Tool 执行
   ├── 不递归调用 Agent
   ├── config 显式透传或隔离
   └── output 压缩
   ↓
Scratchpad Manager
   ├── 最近 N 步
   ├── 长结果摘要
   └── token budget 检查
   ↓
Final Response
```

## 一句话判断

* **config 丢失**：wrapper 没透传 `config`
* **callback 溢出**：tool 内部递归调用 agent/chain 且 callbacks 复用失控
* **scratchpad 爆炸**：ReAct 中间步骤和工具结果未裁剪、未摘要、未外存化

最实用的组合是：**显式透传 config + tool 内隔离 callbacks + scratchpad 只保留最近 3–5 步 + 大结果外存化**。

# 实体识别怎么做？什么BERT模型？
## 实体识别（Named Entity Recognition，简称 NER）​ 是自然语言处理（NLP）中的一个基础任务，指的是从文本中自动识别出具有特定意义的实体，并给它们分类。
简单来说，就是让计算机像人一样，从一段话里把“关键人名、地名、机构名、时间、疾病、药物”等信息挑出来，并知道它们分别是什么类型。
实体识别（NER）：通常采用序列标注（如BIO/BIOES标签），现在主流是BERT + CRF或BERT + Softmax。
BERT：Bidirectional Encoder Representations from Transformers。是谷歌在2018年提出的一种基于 Transformer 的预训练语言模型，目前在自然语言处理（NLP）里非常基础和常用，你可以把它理解成：一个“先在大文本上学会语言和知识，再拿去干具体任务”的通用语言理解模型。

# 大模型是用本地的还是商用API？你微调过大模型吗？
答：企业通常结合使用。非敏感/通用场景用商用API（如OpenAI、通义千问）快速验证；核心业务/敏感数据用本地私有化部署（如vLLM、TensorRT-LLM）。
微调：具备。包括全量微调（资源消耗大）、LoRA/QLoRA（主流，只训练低秩矩阵，显存占用小）、Prompt Tuning（只调提示词）。

# 大模型微调方法
## 一、全参数微调（Full Fine-Turning）
原理：对预训练模型的所有参数都进行更新。
特点：
最传统、最直接的方法；
在数据充足、算力足够时效果通常最好；
显存和计算成本高，不适合超大模型（如 7B、13B 以上在普通设备上难以承受）；
容易出现灾难性遗忘（忘记预训练阶段学到的通用知识）。
适用场景：模型规模较小（如 BERT-base、DistilBERT），或你有大规模领域数据和高性能算力。
## 二、参数高效微调（Parameter-Efficient Fine-Tuning，PEFT）
这是目前大模型时代最主流的微调思路：只训练很少一部分参数，其余参数冻结。
### 1. 适配器微调（Adapter Tuning）
原理：在 Transformer 的每层（如多头注意力和 FFN 之后）插入小的可训练模块（Adapter 层），训练时只更新 Adapter 参数，主体模型参数冻结。
优点：参数量增加极少（通常 < 5%），效果好，切换任务只需换 Adapter；
缺点：会增加推理时的前向传播路径，轻微影响延迟。
### 2. 前缀微调 / 提示微调（Prefix Tuning / Prompt Tuning）
Prefix Tuning：在输入层或每一层的隐藏状态前添加可训练的 前缀向量（prefix），只训练这些向量，模型其余参数冻结。
Prompt Tuning：类似，但只在输入嵌入层添加可训练的 prompt embedding（可以看作 Prefix 的简化版）。
特点：
参数量极小；
对大模型效果接近全微调，对小模型效果可能略差；
不直接修改模型结构。
### 3. LoRA（Low-Rank Adaptation，目前非常流行）
原理： 不直接微调原始的庞大权重矩阵，而是通过低秩分解引入一小部分可训练参数来近似权重的更新量
假设权重更新矩阵 ΔW 是低秩的，用两个小矩阵 A、B（低秩分解）来近似 ΔW，训练时只更新 A、B，原始权重 W 冻结。
优点：
参数量极少（如原模型的 1%~3%）；
不增加推理额外路径（训练完可将 A×B 合并回原权重）；
在 7B~70B 模型上效果接近全微调，显存占用大幅下降；
工具支持：HuggingFace PEFT 库原生支持，是目前实操中最常用的微调方式之一。
### 4. QLoRA（Quantized LoRA）
原理：在 LoRA 基础上，把预训练模型量化为 4-bit（NF4 / FP4），再用 LoRA 微调。
优点：
能在消费级显卡（如单张 24GB 3090/4090）上微调 13B~70B 级别模型；
几乎不损失性能；
地位：当前个人/中小团队微调超大模型的首选方案之一。
## 三、部分参数微调（Partial / Layer-wise Fine-Tuning）
原理：只微调模型的顶层若干层（如最后 2~4 层 Transformer），或只微调特定模块（如分类头、LayerNorm、Attention 的某些参数）。
动机：底层学到的是通用语言/语法知识，顶层更贴近任务语义，只调顶层可节约算力、减缓遗忘。
特点：简单直接，但效果通常不如 LoRA / Adapter 等系统化方法稳定。
## 四、指令微调（Instruction Tuning）
这不是一种“技术结构”，而是一种数据和使用方式：
用 （指令, 输入, 输出）​ 格式的数据集（如 Alpaca、FLAN）继续微调模型；
技术上可以用全微调、LoRA、QLoRA​ 等任意方式实现；
目的是让模型更好地理解人类指令，完成对话、问答、抽取等任务。
你如果用 BERT/BioBERT 做 NER、分类，一般不说“指令微调”，这个概念更多用在 GPT 类生成式大模型（LLaMA、ChatGLM、Baichuan 等）。

## 部署用docker还是k8s？
答：两者结合。Docker负责单容器的打包与环境隔离；Kubernetes (k8s)负责多容器的编排、自动扩缩容、负载均衡和故障自愈。生产环境通常是 Docker + k8s。

## RAG基础知识、范式？自己搭建还是直接用API？
范式：Retrieval（检索） -> Augmented（增强） -> Generation（生成）。核心是打破大模型知识截止性。
搭建 vs API：简单应用可直接用LangChain/LLamaIndex + 向量数据库API；复杂企业级、高定制化场景建议自研（如自研切片、重排、路由逻辑），以确保数据安全和效果最优。

## Transformer架构？需要优化的点？
架构：基于自注意力机制（Self-Attention），包含多头注意力、前馈网络、残差连接和层归一化。
优化点：
计算复杂度：O(n2)的平方复杂度，长文本（>4k）推理慢，优化方案如 Flash Attention、Longformer（稀疏注意力）。
显存占用：大模型显存压力大，优化如梯度检查点（Gradient Checkpointing）、混合精度（FP16/BF16）。
位置编码：绝对位置编码限制长度，优化为 RoPE（旋转位置编码，如LLaMA）或 ALiBi。

## RAG优化（实现细节与效果）
RAG优化不是单一的，而是全链路的优化。主要从四个维度入手：
### Query侧优化（查询理解与扩展）：
Query Rewrite（查询改写）：利用LLM将用户的口语化问题改写为更适合检索的结构化Query（如“怎么退费” -> “退款流程步骤及政策”）。
Multi-Query（多路查询）：将一个问题拆解成多个子问题，分别检索后合并去重，提升召回率。
### 检索侧优化（提升召回质量）：
多路召回（Hybrid Search）：向量检索（语义）+ 稀疏检索（BM25/关键词）+ 知识图谱（关系）(Knowledge Graph，KG)：是一种用“实体—关系—实体”结构化表示知识的数据库，关注的是知识本身是怎么组织成的。
重排序（Rerank）：粗排Top 100后，使用 Cross-Encoder（如 BGE-Reranker）进行精排，剔除无关噪音，只保留Top 5送给LLM，大幅提升准确率。
### Context/索引侧优化（上下文工程）：
智能分块（Chunking）：根据文档结构或语义边界分块，避免切分句子。
元数据过滤（Metadata Filtering）：在检索时加入时间、类别、部门等元数据过滤，缩小搜索范围。
### 生成侧优化（提升答案质量）：
Prompt Engineering：在Prompt中加入“若检索信息不足，请明确回复不知道，禁止编造”，降低幻觉。
Self-RAG/Corrective RAG：让模型在生成后自我反思，迭代修正答案。
## RAG评估指标
评估不能只看最终答案，必须分层评估：
检索层（Retrieval Metrics）：评估“找得对不对”。
Recall@K：前K个结果中包含正确文档的比例。
MRR (Mean Reciprocal Rank)：第一个正确文档排名的倒数平均值。
NDCG：综合考虑文档相关性和排序位置的指标。
生成层（Generation Metrics）：评估“答得好不好”。
Faithfulness（忠实度）：答案是否严格基于检索到的上下文，无幻觉。
Answer Relevance（答案相关性）：答案是否与用户问题相关。
Answer Correctness（答案正确性）：答案与标准答案的语义一致性（常用BERTScore、F1、Exact Match）。
系统层（System Metrics）：
Latency（延迟）：检索+生成的总耗时（通常要求<2s）。
Cost（成本）：Token消耗量。
### 多路召回方案
多路召回的核心是“不把鸡蛋放在一个篮子里”，通过异构检索方式互补短板。
典型三路召回：
稀疏检索（Sparse）：如 BM25、TF-IDF。擅长精确匹配（产品型号、专有名词、数字），但对语义不敏感。
稠密检索（Dense）：如 Embedding + 向量数据库（Milvus/Faiss）。擅长语义相似（同义词、上位词），但对精确词匹配差。
结构化/知识图谱召回：基于实体关系、SQL规则。擅长逻辑推理（如多跳问答）。
结果融合策略：
RRF (Reciprocal Rank Fusion)：业界首选。不依赖原始分数，仅根据各路内部的排名计算得分（score=∑1/(k+rank)），鲁棒性强，无需调参。
加权得分：α×向量分+β×BM25分（需归一化和调参）。
重排序（Rerank）：多路召回后，使用 Cross-Encoder 对候选集进行精排，选出最相关的 Top-K 送入大模型。
## 分块方案（Chunking）
分块是决定RAG效果的基石，直接决定了检索的粒度。
固定大小分块（Fixed-size）：按固定Token数（如512）切分，配合 Overlap（重叠10%-20%）防止断句。优点是实现简单；缺点是容易切断语义。
语义分块（Semantic）：通过计算句子Embedding的余弦相似度，在语义边界处切开。优点是语义完整；缺点是计算成本高。
递归分块（Recursive）：先按段落切，段落太大再按句子切，最后按字符切。尽可能保留高级别语义结构，工业界常用。
基于文档结构分块（Structure-aware）：利用Markdown标题、HTML标签、PDF章节等进行切分。适合结构化文档（如技术手册、法律合同）。
后期分块（Late Chunking）：先让长上下文模型（如Gemini 1.5）处理全文生成向量，再做物理切分。让小块携带全局上下文特征，解决“见木不见林”问题。

# 面经

## agnet怎么调工具
结构化调用, 模型生成function_call结构化参数, agent拿到后解析json, 调用真实函数拿到执行结果, 返回给模型。

本项目用的是 OpenAI Function Calling：模型只“提议”调哪个工具，真正执行在本地 handler
run() → 循环 step()
         ├─ _think()   ：把 tools schema 发给 LLM，拿回 tool_calls
         └─ _execute() ：按名字找 ToolSpec.handler 并执行，结果写回 messages

*具体流程*
1. 工具怎么交给模型
_think() 里调用 chat_completion，第三个参数是工具说明书：
assistant = chat_completion(
    self.endpoint,
    api_messages,
    self._openai_tools(),  # [ToolSpec.openai_schema, ...]
_openai_tools() 从 self.tool_specs 抽出 schema（如 KnowledgeTool 的参数定义）。llm_client 把它们放进请求体的 tools 字段。

2. 模型怎么“调”
模型不直接跑代码，而是在回复里带 tool_calls，例如：

{
  "tool_calls": [{
    "id": "call_xxx",
    "function": {
      "name": "KnowledgeTool",
      "arguments": "{\"kbsId\":\"...\",\"query\":\"...\"}"
    }
  }]
}
normalize_tool_calls 把它统一成 {id, name, arguments}。

3. Agent 怎么真正执行
_execute()：

用工具名查 ToolSpec（openai_function_name_to_spec）
parse_tool_arguments 把 JSON 字符串解析成 dict
调用 spec.handler(**args)
结果以 role: "tool" 写回 self.messages，下一轮再给模型看
            args = parse_tool_arguments(str(tc.get("arguments", "{}")))
            spec = spec_map.get(fn)
            ...
                    result = spec.handler(**args)
例如 KnowledgeTool → knowledge_query → rag.similarity_search(...)。

4. 工具从哪来
tools.py：build_default_tool_specs 组装 schema + handler
factory.py / biz.py：按 Agent 的 allowed_tools 筛出运行时工具，塞进 JChatMind.tool_specs

## 还有什么调用姿势(指的是 Agent 让模型用外部能力的几种常见方式)
姿势	|模型产出什么|	谁执行	|特点
结构化 Function Calling	tool_calls |JSON（名字+参数）|	Agent 按名字调本地函数	|你项目用的就是这个
ReAct	|文本里交替写 Thought / Action / Observation|	Agent 解析文本再调工具	|边想边做，可解释，但解析不如 JSON 稳
Plan-and-Execute|	先出完整计划，再逐步执行|	Planner + Executor 分工	|适合多步任务，计划可改
代码解释器	|一段可运行代码（常是 Python）|	沙箱(一块隔离、受限的执行环境：代码能跑，但动不了你真正在乎的东西)里跑代码|	通用能力强：算数、画图、读表
MCP	|按协议发现/调用工具|	MCP Server 提供工具|	工具可跨应用复用、标准化

## 模型怎么判断要不要调用工具
system prompt与参数引导, 在提示词模板中写可以调用工具; 工具调用参数, 设置为auto自动调用, required必须调用;
训练数据分别, 训练时多少比例的样本走了工具调用, 模型推理时就有多大概率调用工具

项目中请求里 tool_choice="auto"，每轮 _think 带上决策 prompt、知识库列表和 tools schema；模型若返回 tool_calls 就进 _execute，否则视为答完结束。判断在模型侧，框架侧只检测有没有 tool_calls。

## token爆了怎么办
滑动窗口截断 只保留系统提示词和最近N轮对话, 按语义边界截断
摘要压缩 用模型将对话历史压缩成摘要
检索式加载 对话历史存入向量数据库中, 问题来了做向量检索, 只把最相关的放进上下文
长上下文模型 现有模型可支持100-200K上下文, 但是支持不等于能有效利用长上下文

## RAG链路怎么跑
离线阶段 文档解析(pdf,md等用各自的解析方式)->分块(固定分块、语义分块、递归分块...; 注意设置块间重叠, 防止关键信息卡在边界上被截断)->embedding模型转为向量存进向量库

在线阶段 用户提问->做预处理(清洗、纠错、Query Rewrite（改写成更适合检索的问法）、补实体/同义词。作用： 提高召回——用户口语问法往往和文档表述不一致，不处理容易向量对不上)后通过embedding编码->去向量数据库中检索(通过向量相似度)取top-k->因为向量相似度不等于语义相关性, 所以加重排序(向量检索是「近似语义相近」，会混进字面像但不相关的片段。Cross-Encoder 把「问题 + 候选 chunk」成对打分，再筛一遍。作用： 提高精排精度——Top-K 先粗召回，重排再挑真正相关的，减少噪音进 LLM。), 用cross encoder精排后筛选真正相关的, 最后把检索内容加上引用(把命中片段带上文档名、段落 ID 等，和问题一起给模型，并要求回答可溯源。作用： 可解释、可验真、降幻觉；产品上还能点开原文。)和问题一起提交给大模型生成回答

### 补充(HYDE 多路召回)
#### HyDE（Hypothetical Document Embeddings，假设文档嵌入） 是一种 Query 改写 / 检索增强 方法：不直接对用户问题做 embedding，而是先让 LLM 编一段“假设答案文档”，再对这段假文档做 embedding 去向量库检索。
因为用户问题往往短、口语化；知识库里的 chunk 是完整陈述句。直接用问题向量检索，容易和文档向量“对不齐”。HyDE 的假设是：假答案的写法更接近真实文档，用它检索更容易命中相关段落。

流程
用户问题
  → LLM 生成一段“假设会存在于知识库里的答案文本”（可含幻觉，没关系）
  → 对这段假文档做 Embedding
  → 用该向量做相似度检索 Top-K
  → 用真实检索到的文档（不是假文档）交给 LLM 最终回答
注意：假文档只用于 检索，最终回答仍应基于真实命中内容
优缺点
优点： 问法和文档表述差距大时，常能提高召回
缺点： 多一次 LLM 调用（延迟/成本↑）；假文档偏题时会带着检索跑偏

#### 多路召回是检索时不靠单一方法，而是用 几条不同通道各取一批候选，再合并、去重、排序，最后得到更稳的 Top-K
单一检索各有短板：

一路|	擅长	|不擅长
向量（稠密）检索|语义相近、同义改写|专有名词、型号、精确词
BM25（稀疏）检索|关键词、编号、术语|语义、换说法
结构化 / 图谱|关系、多跳逻辑|开放语义问答
多路就是：互补短板，不把鸡蛋放一个篮子里。

流程
用户问题
  ├─ 路1：Embedding → 向量库 Top-K1
  ├─ 路2：BM25 / 关键词 → Top-K2
  └─ 路3：（可选）规则 / SQL / 图谱
        ↓
   合并去重（如 RRF）
        ↓
   （可选）Cross-Encoder 重排
        ↓
   最终 Top-K → 给 LLM

融合
常用 RRF（Reciprocal Rank Fusion）：不看各路原始分数，只按排名加权合并，省得调分、对齐分数尺度。

项目用的是pgvector相似检索(向量（稠密）检索)。

## embedding怎么检索向量
HNSW 构建分层图结构, 上层节点稀疏, 用于快速跳跃到目标区域, 下层节点稠密用于做目标匹配
IVF 先kmean聚类, 查询时只搜索最近的几个聚类中心, 不用扫全量(适合亿级规模场景)
余弦相似度(项目用) 

### 补充数据库选型
Milvus适合企业级生产
qdrant轻量高性能
Chroma适合原型开发
pgvector适合已有Postgresql基础设施

## Agent记忆怎么管
项目里的 Agent「记忆」本质是 **短期对话上下文窗口**，不是独立记忆模块。分三层管：
1. 工作记忆（运行时）
`JChatMind.messages`：当前这次跑 Agent 时塞给 LLM 的上下文。
- 用户 / assistant / tool 消息不断 `append`
- 每轮后用 `_trim_messages()` 裁剪，避免无限涨
策略：**保留首条 system，再只留最近 N 条**（`max_messages`，默认约 20，可由 `chatOptions.messageLength` 配）。

2. 长期落库（会话级）
每条产出经 `save_message` 写入 PostgreSQL `chat_message`（session 维度）。
下次同会话再聊时：
```
DB 取近期消息 → default_messages_from_history() → 拼进 messages
```
即：**长期记在库里，短期只加载最近一段进上下文。**

3. 外部知识（不算会话记忆）
知识库 / RAG 通过 `KnowledgeTool` 按需检索，属于**外部记忆**，不塞进常驻 history。
---
面试一句话

| 类型 | 你项目怎么做 |
|---|---|
| 短期记忆 | `messages` + 滑动窗口裁剪 |
| 长期记忆 | `chat_message` 表持久化，启动时加载近期 |
| 外部记忆 | RAG / 工具，按需查 |

**没有**做：向量化记忆、摘要压缩（summary memory）、实体记忆等更高级方案。  
管法就是：**DB 全量存会话，推理时只喂 system + 最近 N 条。**